# Homework 01: Overfitting and How to Fix It

**Student Version**

**Release:** May 21, 2026  
**Deadline:** May 24, 2026  
**Recommended time budget:** 3-5 focused hours  
**Recommended environment:** Google Colab or a local Python environment with PyTorch and torchvision. A GPU is helpful but not required for the required path.

In this homework you will build a working image classifier, intentionally make it overfit, and then improve it with regularization. The goal is not to chase the best possible FashionMNIST score. The goal is to show that you can diagnose training behavior and make a fair comparison between experiments.

Goals for this homework:
- Build a full PyTorch training pipeline on **FashionMNIST**
- Train a baseline model and reach a reasonable test-set result
- Demonstrate **overfitting** with clear evidence
- Reduce overfitting using methods from class
- Perform error analysis and summarize your experiments

Minimum viable submission:
- one baseline MLP experiment
- one deliberately overfitting experiment
- one regularized/improved experiment
- learning curves for the main experiments
- confusion matrix or misclassification examples for your best model
- a short summary table and written answers

Suggested success criteria:
- **Task 1:** reach about **86% test accuracy** with a baseline MLP; slightly lower is acceptable if the experiment is correct and analyzed honestly
- **Task 2:** show a clear train/validation gap, or validation loss getting worse while training loss improves
- **Task 3:** reduce the gap and/or improve validation/test behavior using at least two regularization choices

Grading guide:

| Part | Weight | What matters most |
|---|---:|---|
| Task 1: baseline | 20% | working training loop, metrics, learning curves, test accuracy |
| Task 2: overfitting evidence | 25% | convincing train/validation evidence and explanation |
| Task 3: fix | 25% | fair comparison, at least two improvements, better generalization behavior |
| Task 4: error analysis | 15% | confusion matrix or mistakes, useful interpretation |
| Task 5 + written answers | 15% | clear summary table, concise conclusions, reproducibility |

Keep your notebook readable. A grader should be able to rerun it from top to bottom after you remove or resolve the `NotImplementedError` cells.


## 0. Setup

The helper code below is provided to keep the homework focused on modeling, experiments, and analysis rather than boilerplate.

In [ ]:
import copy
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class TransformedSubset(Dataset):
    def __init__(self, base_dataset, indices, transform=None):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, label = self.base_dataset[self.indices[idx]]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def build_loaders(train_transform, eval_transform, batch_size=128, val_size=5000, train_limit=None, seed=42):
    train_base = datasets.FashionMNIST(root='data', train=True, download=True, transform=None)
    test_base = datasets.FashionMNIST(root='data', train=False, download=True, transform=None)

    generator = torch.Generator().manual_seed(seed)
    all_indices = torch.randperm(len(train_base), generator=generator).tolist()
    val_indices = all_indices[:val_size]
    train_indices = all_indices[val_size:]
    if train_limit is not None:
        train_indices = train_indices[:train_limit]

    train_ds = TransformedSubset(train_base, train_indices, transform=train_transform)
    val_ds = TransformedSubset(train_base, val_indices, transform=eval_transform)
    test_ds = TransformedSubset(test_base, range(len(test_base)), transform=eval_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader, train_base.classes


def show_batch(loader, class_names, max_items=8):
    images, labels = next(iter(loader))
    max_items = min(max_items, len(images))
    fig, axes = plt.subplots(1, max_items, figsize=(1.8 * max_items, 2.6))
    if max_items == 1:
        axes = [axes]
    for ax, image, label in zip(axes, images[:max_items], labels[:max_items]):
        ax.imshow(image.squeeze(0), cmap='gray')
        ax.set_title(class_names[label])
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def accuracy_from_logits(logits, targets):
    predictions = logits.argmax(dim=1)
    return (predictions == targets).float().mean().item()


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_examples += labels.size(0)

    return {
        'loss': total_loss / total_examples,
        'acc': total_correct / total_examples,
    }


def fit_model(model, train_loader, val_loader, criterion, optimizer, epochs):
    history = []
    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_metrics = run_epoch(model, val_loader, criterion, optimizer=None)
        history.append({
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'train_acc': train_metrics['acc'],
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['acc'],
        })
        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['acc']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['acc']:.4f}"
        )
    return pd.DataFrame(history)


def evaluate_model(model, loader, criterion):
    return run_epoch(model, loader, criterion, optimizer=None)


def plot_history(history_df, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train loss')
    axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val loss')
    axes[0].set_title(f'{title}: loss')
    axes[0].set_xlabel('epoch')
    axes[0].legend()

    axes[1].plot(history_df['epoch'], history_df['train_acc'], label='train acc')
    axes[1].plot(history_df['epoch'], history_df['val_acc'], label='val acc')
    axes[1].set_title(f'{title}: accuracy')
    axes[1].set_xlabel('epoch')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def collect_predictions(model, loader):
    model.eval()
    all_images, all_labels, all_preds = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(device))
            preds = logits.argmax(dim=1).cpu()
            all_images.append(images.cpu())
            all_labels.append(labels.cpu())
            all_preds.append(preds)
    return torch.cat(all_images), torch.cat(all_labels), torch.cat(all_preds)


def show_misclassifications(images, labels, preds, class_names, max_items=8):
    wrong = torch.nonzero(labels != preds).squeeze(1)
    if len(wrong) == 0:
        print('No mistakes found.')
        return
    max_items = min(max_items, len(wrong))
    fig, axes = plt.subplots(1, max_items, figsize=(2.2 * max_items, 3.0))
    if max_items == 1:
        axes = [axes]
    for ax, idx in zip(axes, wrong[:max_items]):
        ax.imshow(images[idx].squeeze(0), cmap='gray')
        ax.set_title(f'T: {class_names[labels[idx]]}\nP: {class_names[preds[idx]]}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()


set_seed(42)


## Helper Function Contracts

The setup cell above defines helper functions so you do not have to write all boilerplate from scratch. Use them whenever a task asks you to create loaders, train a model, plot curves, evaluate, or inspect mistakes.

| Helper | Use it when | Inputs | Returns |
|---|---|---|---|
| `build_loaders(train_transform, eval_transform, batch_size=128, val_size=5000, train_limit=None, seed=42)` | You need FashionMNIST train/validation/test loaders | training transform, evaluation transform, batch size, validation size, optional train subset size, seed | `train_loader, val_loader, test_loader, class_names, train_size` |
| `show_batch(loader, class_names, max_items=8)` | You want to inspect a few images from a loader | data loader, class name list, number of images | displays a figure, returns nothing |
| `fit_model(model, train_loader, val_loader, criterion, optimizer, epochs)` | You need to train a model and track metrics | model, train/validation loaders, loss, optimizer, epoch count | pandas `DataFrame` with `epoch`, `train_loss`, `train_acc`, `val_loss`, `val_acc` |
| `evaluate_model(model, loader, criterion)` | You need final validation/test metrics | model, loader, loss | dictionary with `loss` and `acc` |
| `plot_history(history_df, title)` | You need learning curves | history DataFrame from `fit_model`, plot title | displays loss and accuracy curves, returns nothing |
| `collect_predictions(model, loader)` | You need predictions for error analysis | trained model, loader | `images, labels, preds` tensors on CPU |
| `show_misclassifications(images, labels, preds, class_names, max_items=8)` | You need to view wrong predictions | tensors from `collect_predictions`, class names, number of examples | displays misclassified examples, returns nothing |

You can still write your own code if you want, but the required path is designed around these helpers.


## 1. Load FashionMNIST and inspect the data

### Exercise 1.1
Define your training and evaluation transforms, create loaders, and inspect one batch.

Required:
- use `transforms.ToTensor()` for the baseline transform
- create train, validation, and test loaders with `build_loaders(...)`
- print or display the class names
- visualize one small batch with `show_batch`

Hints:
- Use the **full training split** for Task 1.
- For Tasks 2 and 3 you may use a smaller training subset, such as 4k or 5k images, to make overfitting easier to observe and keep runtime manageable.


In [ ]:
# Exercise 1.1

# TODO:
# 1. define baseline_train_transform and baseline_eval_transform
# 2. create baseline_train_loader, baseline_val_loader, baseline_test_loader, class_names, baseline_train_size
# 3. visualize a batch

# Example starting point:
# baseline_train_transform = transforms.Compose([
#     transforms.ToTensor(),
# ])
# baseline_eval_transform = transforms.Compose([
#     transforms.ToTensor(),
# ])
# baseline_train_loader, baseline_val_loader, baseline_test_loader, class_names, baseline_train_size = build_loaders(
#     train_transform=baseline_train_transform,
#     eval_transform=baseline_eval_transform,
#     batch_size=128,
#     val_size=5000,
#     train_limit=None,
#     seed=42,
# )
# show_batch(baseline_train_loader, class_names)
# print('Training examples:', baseline_train_size)

raise NotImplementedError('Fill in Exercise 1.1')


## 2. Task 1: Build a baseline model

Build a **baseline MLP** and train it end-to-end.

Requirements:
- Use a clean training loop
- Track **train** and **validation** metrics per epoch
- Plot learning curves with `plot_history(...)`
- Report final **test accuracy** with `evaluate_model(...)`
- Try to reach around **86%** test accuracy

Keep the model modest. A good baseline for this homework is a small MLP with `Flatten -> Linear -> ReLU -> Linear -> ReLU -> Linear`.

Use `fit_model(...)` for training unless you intentionally want to write your own loop. Do not spend too long tuning this model. If your code is correct and your result is close, move on to the overfitting and regularization comparison.


In [ ]:
# Task 1

# TODO:
# 1. define a baseline model class
# 2. instantiate the model on `device`
# 3. define criterion and optimizer
# 4. train for several epochs with fit_model(...)
# 5. plot history with plot_history(...)
# 6. evaluate on the test set with evaluate_model(...)

# Suggested variables to create:
# baseline_model
# baseline_criterion
# baseline_optimizer
# baseline_history
# baseline_test_metrics
# baseline_gap

raise NotImplementedError('Implement Task 1')


In [ ]:
# Task 1 sanity check
# Run this after you finish Task 1.

required_history_columns = {'epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc'}
assert isinstance(baseline_history, pd.DataFrame), 'baseline_history should be a pandas DataFrame'
assert required_history_columns.issubset(baseline_history.columns), baseline_history.columns.tolist()
assert isinstance(baseline_test_metrics, dict), 'baseline_test_metrics should be a dictionary'
assert 'acc' in baseline_test_metrics, 'baseline_test_metrics should contain an accuracy value under key "acc"'
assert 0.0 <= baseline_test_metrics['acc'] <= 1.0
print('Baseline final validation accuracy:', round(float(baseline_history.iloc[-1]['val_acc']), 4))
print('Baseline test accuracy:', round(float(baseline_test_metrics['acc']), 4))


## 3. Task 2: Demonstrate overfitting

Now create a setup that **clearly overfits**.

Ideas:
- use a larger model
- train longer
- remove regularization
- optionally use a **smaller training subset** such as 4k or 5k images

Your goal is not just to say that the model overfits. Your goal is to **show evidence**:
- train accuracy noticeably above validation accuracy
- and/or validation loss rising while training loss keeps falling

Use `build_loaders(...)` again with the same `val_size` and `seed`, but with a smaller `train_limit`. Keep the validation split unchanged so the comparison is meaningful. If your gap is still small, reduce the training subset further, for example to 3k images, or train for a few more epochs.


In [ ]:
# Task 2

# TODO:
# 1. create loaders for the overfitting experiment with build_loaders(..., train_limit=4000, seed=42)
# 2. define a larger model (without Dropout / BatchNorm)
# 3. train it long enough to produce a clear train/val gap with fit_model(...)
# 4. plot the curves with plot_history(...)
# 5. compute a simple overfitting gap such as:
#    final_train_acc - final_val_acc

# Suggested variables to create:
# overfit_train_loader, overfit_val_loader, overfit_test_loader, overfit_train_size
# overfit_model
# overfit_history
# overfit_test_metrics
# overfit_gap

raise NotImplementedError('Implement Task 2')


### Task 2 written interpretation

Write 3-5 sentences here after running your overfitting experiment:

- What is the final train/validation gap?
- Which curve makes the overfitting visible?
- Why is this a real overfitting example rather than just a low-quality model?


## 4. Task 3: Fix the model

Start from the overfitting setup and improve it.

Use at least **two** of the following:
- Dropout
- weight decay
- data augmentation
- smaller model capacity
- early stopping or a shorter training budget

Requirements:
- keep the comparison fair: same dataset split, similar training budget, clear description of changes
- explain what you changed
- show that the gap becomes smaller and/or validation/test behavior improves

Use the same helper pattern as Task 2: `build_loaders(...)`, `fit_model(...)`, `plot_history(...)`, and `evaluate_model(...)`. It is okay if the improved model has slightly lower training accuracy. Better generalization is the point.


In [ ]:
# Task 3

# TODO:
# 1. define improved transforms and / or model changes
# 2. create loaders with the same train_limit and seed as Task 2
# 3. train a regularized version of the model with fit_model(...)
# 4. compare its history to the overfitting history with plot_history(...)
# 5. evaluate on the test set with evaluate_model(...)

# Suggested variables to create:
# fixed_train_loader, fixed_val_loader, fixed_test_loader, fixed_train_size
# fixed_model
# fixed_history
# fixed_test_metrics
# fixed_gap

raise NotImplementedError('Implement Task 3')


In [ ]:
# Task 3 comparison check
# Run this after you finish Task 3.

print('Overfit gap:', round(float(overfit_gap), 4))
print('Fixed gap  :', round(float(fixed_gap), 4))
print('Overfit test accuracy:', round(float(overfit_test_metrics['acc']), 4))
print('Fixed test accuracy  :', round(float(fixed_test_metrics['acc']), 4))

if fixed_gap <= overfit_gap:
    print('Good: the regularized model reduced the train/validation gap.')
else:
    print('Warning: the gap did not shrink. Explain why, or revisit your regularization choices.')


### Task 3 written interpretation

Write 3-5 sentences here:

- Which two or more changes did you make?
- Which metric improved most?
- Did the model generalize better? Use numbers from your experiment.


## 5. Task 4: Error analysis

Use your **best final model** and inspect what it still gets wrong.

Required pieces:
- confusion matrix, using `confusion_matrix(...)` and `ConfusionMatrixDisplay(...)`
- at least 6 misclassified examples, using `collect_predictions(...)` and `show_misclassifications(...)`
- short interpretation of common mistakes

Look for classes that seem visually similar or ambiguous. For example, FashionMNIST often confuses shirts, T-shirts, coats, and pullovers.


In [ ]:
# Task 4

# TODO:
# 1. collect predictions from your best final model
# 2. plot a confusion matrix
# 3. show several misclassified examples

# Example helper calls:
# images, labels, preds = collect_predictions(fixed_model, fixed_test_loader)
# cm = confusion_matrix(labels.numpy(), preds.numpy())
# ConfusionMatrixDisplay(cm, display_labels=class_names).plot(xticks_rotation=45)
# show_misclassifications(images, labels, preds, class_names, max_items=8)

raise NotImplementedError('Implement Task 4')


### Task 4 written interpretation

Write 3-5 sentences here:

- Which classes were most often confused?
- Do the mistakes look visually ambiguous?
- What change might reduce these errors?


## 6. Task 5: Experiment summary

Create one summary table with one row per main experiment.

Suggested columns:
- experiment name
- train subset size
- key changes
- final train accuracy
- final validation accuracy
- test accuracy
- train/validation gap
- short conclusion

This section should make it easy for a reader to understand your work without reading every code cell.


In [ ]:
# Task 5

# TODO: create a small DataFrame summarizing your main experiments

raise NotImplementedError('Implement Task 5')


## 7. Homework deliverable

Before submitting, check that your final notebook contains:
- [ ] a working baseline experiment
- [ ] one clear overfitting experiment
- [ ] one improved / regularized experiment
- [ ] learning curves for the main experiments
- [ ] confusion matrix and at least 6 misclassified examples
- [ ] one final summary table
- [ ] short written answers to the wrap-up questions below
- [ ] no unresolved `NotImplementedError` cells in the required path

## 8. Wrap-up questions

Please answer briefly in markdown:

1. What evidence showed that your model was overfitting?
2. Which changes helped the most, and why do you think they helped?
3. Which class pairs were hardest for the model to separate?
4. If you had one extra hour, what would you try next?

## Optional extension

After the next class, you may optionally compare your best MLP to a small CNN. This is **not required** for the core homework and should not replace the required MLP comparison.
